EXERCICE 1 : Métriques de performance — Régression et Classification
=====================================================================
Objectif : Calculer, comparer et interpréter les métriques de performance
sur un problème de fréquence sinistre (régression) et un problème de 
détection de fraude (classification).

Ce que vous allez apprendre :
- Calculer MSE, MAE, Poisson deviance et R² pour la régression
- Construire et interpréter une matrice de confusion
- Calculer precision, recall, F1 et AUC-ROC pour la classification
- Comprendre l'impact du seuil de décision sur les métriques
- Utiliser make_scorer pour des métriques custom

Pré-requis : M03_F01, M03_F05
Temps estimé : 15 minutes

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import PoissonRegressor, LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score, mean_poisson_deviance,
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score, roc_curve,
    precision_recall_curve, average_precision_score, make_scorer
)
import matplotlib.pyplot as plt

## PARTIE 1 : MÉTRIQUES DE RÉGRESSION (Fréquence sinistre)

In [ ]:
# =============================================================
# PARTIE 1 : MÉTRIQUES DE RÉGRESSION (Fréquence sinistre)
# =============================================================
print("=" * 60)
print("PARTIE 1 : Métriques de régression — Fréquence sinistre")
print("=" * 60)

df = pd.read_csv("../M03_F01_Machine_Learning_Fondamentaux/freMTPL2freq.csv")
features = ['DrivAge', 'BonusMalus', 'VehAge', 'Density']
X = df[features].astype(float)
y = df['ClaimNb'].astype(float)
exposure = df['Exposure'].astype(float)

X_train, X_test, y_train, y_test, exp_train, exp_test = train_test_split(
    X, y, exposure, test_size=0.3, random_state=42
)

freq_train = y_train / exp_train
freq_test = y_test / exp_test

# Entraîner un GLM Poisson
glm = PoissonRegressor(alpha=0, max_iter=1000)
glm.fit(X_train, y_train, sample_weight=exp_train)
freq_pred = np.clip(glm.predict(X_test) / exp_test, 1e-6, None)

# TODO : Calculer les métriques de régression suivantes :
# 1. MSE (Mean Squared Error)
# 2. RMSE (Root Mean Squared Error)
# 3. MAE (Mean Absolute Error)
# 4. R² (coefficient de détermination)
# 5. Poisson Deviance (mean_poisson_deviance)

# Indice : utiliser les fonctions de sklearn.metrics



# TODO : Quelle métrique est la plus adaptée pour un problème de fréquence ?
# Pourquoi la Poisson deviance est-elle préférable à la MSE ici ?

## PARTIE 2 : MÉTRIQUES DE CLASSIFICATION (Détection de fraude)

In [ ]:
# =============================================================
# PARTIE 2 : MÉTRIQUES DE CLASSIFICATION (Détection de fraude)
# =============================================================
print("\n" + "=" * 60)
print("PARTIE 2 : Métriques de classification — Fraude")
print("=" * 60)

# Simuler un dataset de fraude (déséquilibré : 3% de fraude)
np.random.seed(42)
n_claims = 5000
X_fraud = pd.DataFrame({
    'claim_amount': np.random.lognormal(7, 1.5, n_claims),
    'policy_age': np.random.uniform(0, 20, n_claims),
    'claim_delay': np.random.exponential(30, n_claims),
    'n_previous_claims': np.random.poisson(1, n_claims),
    'driver_age': np.random.normal(45, 12, n_claims).clip(18, 80),
})

# Fraude : 3% des cas, corrélée avec montant élevé et délai court
fraud_prob = 1 / (1 + np.exp(-(
    -4
    + 0.5 * (X_fraud['claim_amount'] > 5000).astype(float)
    + 0.3 * (X_fraud['claim_delay'] < 10).astype(float)
    + 0.2 * X_fraud['n_previous_claims']
    + np.random.normal(0, 0.5, n_claims)
)))
y_fraud = (np.random.random(n_claims) < fraud_prob).astype(int)
print(f"Taux de fraude : {y_fraud.mean()*100:.1f}%")

X_fr_train, X_fr_test, y_fr_train, y_fr_test = train_test_split(
    X_fraud, y_fraud, test_size=0.3, random_state=42, stratify=y_fraud
)

# Entraîner un classifieur
clf = GradientBoostingClassifier(n_estimators=100, max_depth=3, random_state=42)
clf.fit(X_fr_train, y_fr_train)

y_pred_class = clf.predict(X_fr_test)
y_pred_proba = clf.predict_proba(X_fr_test)[:, 1]

# TODO : Calculer l'accuracy. Est-elle un bon indicateur ici ?
# Indice : que vaudrait l'accuracy d'un modèle qui prédit TOUJOURS "pas de fraude" ?



# TODO : Afficher la matrice de confusion
# Indice : confusion_matrix(y_fr_test, y_pred_class)
# Identifier : True Positives, False Positives, True Negatives, False Negatives



# TODO : Calculer precision, recall et F1
# Quel est le coût d'un False Negative (fraude non détectée) ?
# Quel est le coût d'un False Positive (enquête inutile) ?



# TODO : Calculer l'AUC-ROC et tracer la courbe ROC
# Indice : roc_auc_score(y_fr_test, y_pred_proba)
#           fpr, tpr, thresholds = roc_curve(y_fr_test, y_pred_proba)

## PARTIE 3 : IMPACT DU SEUIL DE DÉCISION

In [ ]:
# =============================================================
# PARTIE 3 : IMPACT DU SEUIL DE DÉCISION
# =============================================================
print("\n" + "=" * 60)
print("PARTIE 3 : Impact du seuil de décision")
print("=" * 60)

# TODO : Tester différents seuils (0.1, 0.2, 0.3, 0.5, 0.7)
# Pour chaque seuil, calculer precision et recall
# Tracer la courbe precision-recall
# Quel seuil choisiriez-vous pour la fraude ? (compromis coût FP vs FN)

thresholds_to_test = [0.05, 0.1, 0.2, 0.3, 0.5, 0.7]

## PARTIE 4 : MAKE_SCORER — MÉTRIQUE CUSTOM

In [ ]:
# =============================================================
# PARTIE 4 : MAKE_SCORER — MÉTRIQUE CUSTOM
# =============================================================
print("\n" + "=" * 60)
print("PARTIE 4 : Métrique custom (Poisson deviance en CV)")
print("=" * 60)

# TODO : Créer un scorer custom pour la Poisson deviance
# et l'utiliser dans cross_val_score
# Indice :
#   def poisson_deviance_scorer(y_true, y_pred):
#       y_pred = np.clip(y_pred, 1e-6, None)
#       return mean_poisson_deviance(y_true, y_pred)
#   scorer = make_scorer(poisson_deviance_scorer, greater_is_better=False)

## QUESTIONS DE RÉFLEXION

In [ ]:
# =============================================================
# QUESTIONS DE RÉFLEXION
# =============================================================
"""
1. Pour la fréquence sinistre, pourquoi la Poisson deviance est-elle
   préférable au R² ou à la MSE ?

2. L'accuracy de votre modèle de fraude est probablement > 95%.
   Est-ce un bon modèle pour autant ? Pourquoi l'accuracy est trompeuse ?

3. En détection de fraude, préférez-vous maximiser la precision ou le recall ?
   Quel est l'impact business de chaque choix ?

4. Le coefficient de Gini en assurance = 2*AUC - 1. Si votre AUC = 0.75,
   quel est votre Gini ? Est-ce un bon score en pratique ?

5. Pourquoi le choix du seuil est-il un choix MÉTIER et pas un choix 
   technique ? Qui devrait décider du seuil en entreprise ?
"""